## 2. String Normalization & Invalid Value Constraints

This notebook covers:
1. Standardizing noisy string labels (whitespace stripping, case normalization, typo mapping).
2. Setting domain validation rules (replacing negative prices, impossible ages, or invalid ranges with `np.nan` or clipped bounds).

In [4]:
import numpy as np
import pandas as pd

raw_records = {
    'Employee_ID': [1, 2, 3, 4, 5, 6],
    'Department': ['  IT', 'it', 'Sales ', 'FINANCE', 'Sale', 'IT '],
    'Age': [24, -4, 32, 195, 29, 41],                # -4 and 195 are invalid
    'Experience_Years': [2, 0, 8, 12, -1, 15]        # -1 is invalid
}

df = pd.DataFrame(raw_records)
print("=== UNFILTERED RECORDS ===")
display(df)
display(df.dtypes)

=== UNFILTERED RECORDS ===


,Employee_ID,Department,Age,Experience_Years
0,1,IT,24,2
1,2,it,-4,0
2,3,Sales,32,8
3,4,FINANCE,195,12
4,5,Sale,29,-1
5,6,IT,41,15


Employee_ID          int64
Department          object
Age                  int64
Experience_Years     int64
dtype: object

---
## Part 1: String Normalization
- Strip leading/trailing whitespaces: `.str.strip()`
- Standardize case: `.str.upper()` or `.str.lower()`
- Fix typos/abbreviations: `.replace()` using a standardized dictionary

In [7]:
df['Department'] = df['Department'].str.strip().str.lower()
dept_mapping = {
    'sale':'sales',
    'finance':'finance',
    'it': 'information technology'
}
df['Department'] = df['Department'].replace(dept_mapping)
print("=== NORMALIZED CATEGORIES ===")
display(df)

=== NORMALIZED CATEGORIES ===


,Employee_ID,Department,Age,Experience_Years
0,1,information technology,24,2
1,2,information technology,-4,0
2,3,sales,32,8
3,4,finance,195,12
4,5,sales,29,-1
5,6,information technology,41,15


---
## Part 2: Enforcing Domain Constraints

Domain knowledge dictates logical boundaries:
* $18 \le \text{Age} \le 70$
* $\text{Experience\_Years} \ge 0$

Values outside these bounds represent capture errors and should be replaced with `np.nan` so downstream imputers can handle them properly.

In [8]:
df.loc[(df['Age'] < 18) | (df['Age'] > 70), 'Age'] = np.nan

df.loc[df['Experience_Years'] < 0, 'Experience_Years'] = np.nan

display(df)

,Employee_ID,Department,Age,Experience_Years
0,1,information technology,24.0,2.0
1,2,information technology,NaN,0.0
2,3,sales,32.0,8.0
3,4,finance,NaN,12.0
4,5,sales,29.0,NaN
5,6,information technology,41.0,15.0
